# Welcome to Corndel AI6 Workshop 7
## Production Grade Classification

**Level 6 ML Engineer Apprenticeship | Unit 7: Advanced Modelling for Complex Problems**

---

You are stepping into the role of an ML engineer at a mid-sized technology company. The HR director has asked the team to build a model to predict which employees are at risk of leaving. The dataset is real. The problem is harder than it looks.

Work through the cells in order using Shift+Enter. Every run is logged to WandB. By the end of the session you will have a dashboard showing every decision you made and its effect.

---

**What you will do today**

1. Build a naive classifier and discover why accuracy is the wrong metric
2. Fix the class imbalance problem with one parameter change
3. Choose a decision threshold you can professionally justify
4. Use SHAP to understand what the model is actually doing
5. Remove features that should not be in this model and defend that choice
6. Compare all your runs in WandB and tell the story of what you learned

---

## Setup

*Run this cell first. It loads all libraries and helper functions.*

In [ ]:
import os, sys
if not os.path.exists('utils.py'):
    raise FileNotFoundError(
        "\n\nThis notebook requires a companion file called utils.py "
        "in the same folder as this notebook.\n"
        "If you opened this notebook directly without the full repo, "
        "go back and clone or download the complete 7W repository "
        "so that utils.py sits alongside 7w_hr_attrition.ipynb.\n"
        "Expected location: " + os.path.abspath('utils.py')
    )

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import wandb
import xgboost as xgb
import warnings
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, precision_score, recall_score,
)
from utils import (
    plot_class_distribution, plot_confusion_matrix,
    plot_shap_values, plot_roc_calibration,
    plot_threshold_table, log_run, train_clean_model,
)

warnings.filterwarnings('ignore')
plt.rcParams.update({'font.family':'sans-serif',
                     'axes.spines.top':False, 'axes.spines.right':False,
                     'figure.facecolor':'white'})
print('Setup complete.')

### Connect to WandB

Paste your API key when prompted. Every model you train today will be logged automatically.

In [ ]:
wandb.login()
WANDB_PROJECT = 'ai6-7w-hr-attrition'

# If login fails, comment out the two lines above and uncomment this instead:
# wandb.init(mode='disabled'); WANDB_PROJECT = 'ai6-7w-hr-attrition'

print(f'Project: {WANDB_PROJECT}')

---

## Section 1: The Data

*About 20 minutes.*

The IBM HR Analytics dataset contains records for 1,470 employees. Each row is one person. The target column `Attrition` records whether they left the company.

In [ ]:
df = pd.read_csv('ibm_attrition.csv')

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
from utils import plot_class_distribution
plot_class_distribution(df)

### Prepare the features

XGBoost needs numeric inputs. The cells below encode categorical columns and split the data into training and test sets.

**Task:** Three ways to encode Yes/No as 1/0 are shown below. Two will fail or produce wrong results. Uncomment the correct one and run the cell.

In [ ]:
# Uncomment the correct line to turn Yes/No into 1/0.
# The other two produce errors or silent wrong results.

# df['Attrition'] = df['Attrition'].astype(int)         # fails: cannot cast string to int
# df['Attrition'] = df['Attrition'].map({'Yes': 1})      # wrong: No becomes NaN
df['Attrition'] = (df['Attrition'] == 'Yes').astype(int) # correct

print(f"Attrition encoded: {df['Attrition'].value_counts().to_dict()}")

**Task:** Uncomment the correct train/test split. The wrong options either throw an error or produce an unstratified split.

In [ ]:
# Encode remaining categorical columns
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype('category').cat.codes

df = df.drop(['EmployeeCount','EmployeeNumber','Over18','StandardHours'],
              axis=1, errors='ignore')

X = df.drop('Attrition', axis=1)
y = df['Attrition']

# Uncomment the correct split:
# X_train, X_test = train_test_split(X, y, test_size=0.2, random_state=42)          # wrong: unpacks 4 values into 2
# X_train, X_test, y_train, y_test = train_test_split(X, test_size=0.2, random_state=42)  # wrong: missing y
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)  # correct

print(f'Train: {len(X_train)}  Test: {len(X_test)}  Features: {X.shape[1]}')

---

## Section 2: The Naive Model

*About 30 minutes.*

Start with XGBoost at default settings. Do not change any parameters yet. Train it, check the accuracy, note what you see.

In [ ]:
model_naive = xgb.XGBClassifier(
    random_state=42,
    eval_metric='logloss',
    verbosity=0,
)
model_naive.fit(X_train, y_train)
y_pred_naive = model_naive.predict(X_test)

acc = accuracy_score(y_test, y_pred_naive)
print(f'Accuracy: {acc:.1%}')

That accuracy looks reasonable. Before you feel good about it, run the cell below.

<details>
<summary>Before you look: what do you expect the confusion matrix to show?</summary>

You just saw 83% accuracy. That sounds reasonable. But think about the class distribution: around 85% of employees stayed. A model that predicts 'stayed' for every single employee achieves 85% accuracy without learning anything.

What you should expect to see: a large true negative count (correctly predicted stayed), a very small true positive count (correctly predicted left), and a large false negative count (employees who actually left but the model missed). The bottom-left cell is the one to watch.

</details>

In [ ]:
plot_confusion_matrix(y_test, y_pred_naive, title='Naive model')

**Look at the bottom-left cell.** That is the number of employees who actually left but the model predicted would stay. The model has near-perfectly learned to ignore the minority class.

High accuracy, almost zero recall on the class that matters. This is the accuracy trap from 7.2, live.

In [ ]:
# Log the naive run to WandB
log_run(
    project=WANDB_PROJECT,
    run_name='01_naive',
    model=model_naive,
    X_test=X_test,
    y_test=y_test,
    y_pred=y_pred_naive,
    config={'scale_pos_weight': 1, 'threshold': 0.5},
)

---

## Section 3: Fix the Imbalance

*About 30 minutes.*

XGBoost has a parameter called `scale_pos_weight` that tells the model to pay more attention to the minority class. Set it to the ratio of negatives to positives and retrain.

**Task:** Three ways to calculate `scale_pos_weight` are shown below. Only one gives the correct ratio. Uncomment it and run the cell.

In [ ]:
# Uncomment the correct calculation for scale_pos_weight.
# This value tells XGBoost how much extra weight to give each attrition case.

# pos_weight = (y_train == 1).sum() / (y_train == 0).sum()   # wrong: inverted
# pos_weight = len(y_train) / (y_train == 1).sum()            # wrong: total not majority
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()      # correct

print(f'scale_pos_weight = {pos_weight:.1f}')
print(f'Each attrition case now counts as {pos_weight:.0f} stayed cases in training.')

In [ ]:
model_balanced = xgb.XGBClassifier(
    scale_pos_weight=pos_weight,
    random_state=42,
    eval_metric='logloss',
    verbosity=0,
)
model_balanced.fit(X_train, y_train)
y_pred_balanced = model_balanced.predict(X_test)

plot_confusion_matrix(y_test, y_pred_balanced, title='Balanced model')

In [ ]:
# Compare the two models side by side
print(f'{'':20} {'Naive':>10} {'Balanced':>10}')
print('-' * 42)
print(f"{'Accuracy':20} {accuracy_score(y_test, y_pred_naive):>10.1%} "
      f"{accuracy_score(y_test, y_pred_balanced):>10.1%}")
print(f"{'F1 (attrition)':20} "
      f"{f1_score(y_test, y_pred_naive, pos_label=1):>10.3f} "
      f"{f1_score(y_test, y_pred_balanced, pos_label=1):>10.3f}")
print(f"{'ROC AUC':20} "
      f"{roc_auc_score(y_test, model_naive.predict_proba(X_test)[:,1]):>10.3f} "
      f"{roc_auc_score(y_test, model_balanced.predict_proba(X_test)[:,1]):>10.3f}")

In [ ]:
log_run(
    project=WANDB_PROJECT,
    run_name='02_balanced',
    model=model_balanced,
    X_test=X_test,
    y_test=y_test,
    y_pred=y_pred_balanced,
    config={'scale_pos_weight': round(pos_weight, 1), 'threshold': 0.5},
)

---

## Section 4: Choose a Threshold

*About 40 minutes. After lunch.*

The model outputs a probability for each employee. The threshold is the cutoff above which you flag someone as at risk. It is not fixed at 0.5. It is a design decision with consequences for real people.

In [ ]:
y_prob = model_balanced.predict_proba(X_test)[:, 1]
plot_roc_calibration(y_test, y_prob)

<details>
<summary>How to read ROC AUC: what counts as good?</summary>

| ROC AUC | What it means | Example domain |
|:--|:--|:--|
| 0.50 | Random: no better than a coin flip | Lottery prediction |
| 0.60–0.70 | Weak signal, useful but limited | HR attrition, early screening |
| 0.70–0.80 | Moderate, deployable with care | Credit risk scoring |
| 0.80–0.90 | Strong, reliable in most contexts | Fraud detection |
| 0.90+ | Excellent, or suspiciously good | Image classification |

What counts as good depends on the alternative. A 0.63 model for attrition is useful if the alternative is a a manager's gut feeling, which research shows is typically worse. A 0.63 model for cancer screening is dangerous if a specialist radiologist scores 0.95.

The question is always: compared to what, and for whom?

</details>

In [ ]:
plot_threshold_table(y_test, y_prob)

<details>
<summary>What makes a threshold choice defensible?</summary>

A defensible threshold is one you can explain in terms of the people affected, not just the metrics. Anyone can pick 0.3 because the F1 score is higher. That is not a justification.

A defensible answer names the tradeoff explicitly: 'At 0.3, we catch X more employees who are at risk of leaving, at the cost of Y additional false alarms. Given that a false alarm means a retention conversation rather than a serious intervention, we judge that cost to be acceptable.' That reasoning can be written down, reviewed, and challenged. A number cannot.

In a regulated environment, this reasoning would appear in a model card or a deployment decision log, with a named owner.

</details>

### Your threshold decision

Look at the table. There is no right answer. There is only a justified answer.

Think about: if HR acts on a flag, what does that look like for the employee? What is the cost of a false positive? Someone gets a retention conversation they did not need. What is the cost of a false negative? Someone leaves and nobody saw it coming.

Set your chosen threshold in the cell below and justify it in the comment.

**Task:** Three ways to apply a threshold to model probabilities are shown below. Uncomment the correct one. The wrong options either produce booleans instead of integers, or ignore your chosen threshold entirely.

In [ ]:
MY_THRESHOLD = 0.3  # change this to your chosen value
# Reason: 

# Uncomment the correct way to apply the threshold:
# y_pred_threshold = y_prob > MY_THRESHOLD                    # wrong: returns True/False not 0/1
# y_pred_threshold = y_prob.round()                           # wrong: ignores MY_THRESHOLD
y_pred_threshold = (y_prob >= MY_THRESHOLD).astype(int)       # correct

plot_confusion_matrix(y_test, y_pred_threshold,
                      title=f'Balanced model at threshold {MY_THRESHOLD}')

log_run(
    project=WANDB_PROJECT,
    run_name='03_threshold',
    model=model_balanced,
    X_test=X_test,
    y_test=y_test,
    y_pred=y_pred_threshold,
    config={'scale_pos_weight': round(pos_weight,1), 'threshold': MY_THRESHOLD},
)

---

## Section 5: What Is the Model Actually Doing?

*About 45 minutes.*

Accuracy and F1 tell you how well the model performs. SHAP tells you why. It assigns each feature a contribution to each prediction *(see glossary.md)*. Positive values push toward attrition, negative values push away from it.

In [ ]:
plot_shap_values(model_balanced, X_test)

Look at the feature ranking above. Before moving to Section 6:

1. Note the top three features by SHAP importance.
2. Identify which of those are protected characteristics under the Equality Act (Age, Gender, MaritalStatus).
3. Consider whether any non-protected features might indirectly reflect protected characteristics — MonthlyIncome and YearsSinceLastPromotion are worth thinking about.

You will use this reasoning to justify your feature removal choices in Section 6.

---

## Section 6: Feature Selection

*About 20 minutes.*

**The three-question check**

For each feature, ask:

1. Is it a protected characteristic under the Equality Act 2010?
2. Is it special category data under UK GDPR?
3. If you remove it, does model performance drop meaningfully?

If the answer to 1 or 2 is yes, and the answer to 3 is no, the feature should not be in the model. Document the decision either way.

| Feature | Equality Act protected? | GDPR special category? | Keep? |
|:--|:--|:--|:--|
| Age | Yes | No | Discuss |
| Gender | Yes | No | Discuss |
| MaritalStatus | Yes | No | Discuss |
| MonthlyIncome | No | Possibly: may reflect disability or race pay gaps | Discuss |
| YearsSinceLastPromotion | No | Possibly: promotion patterns may correlate with protected groups | Discuss |
| OverTime | No | No | Yes |

*'Possibly' means the feature is not special category data itself but may encode patterns linked to special category characteristics. If removing it does not meaningfully hurt performance, the safer professional choice is to remove it. If it does hurt performance, document the decision and the reasoning. See glossary.md.*

<details>
<summary>What do you remember about DPIAs, and would one be needed here?</summary>

A Data Protection Impact Assessment (DPIA) is a process required under UK GDPR Article 35 when data processing is likely to result in a high risk to individuals. It is not optional when the conditions are met.

A DPIA is specifically required when a system involves systematic and extensive evaluation of personal aspects of individuals using automated processing, where decisions produce legal or similarly significant effects. An HR attrition model that flags employees for retention interventions meets this description.

A DPIA would need to cover: what data is being processed and why, who has access to the outputs, what the risks are to individuals (including indirect discrimination), what safeguards are in place, and who is accountable. It would need to be reviewed before deployment, not after.

In this case, yes: a DPIA would be required before putting this model into production. The feature selection decisions you are making right now are the kind of thing a DPIA would formally document.

</details>

In [ ]:
# Edit this list based on your SHAP analysis above
FEATURES_TO_REMOVE = ['Age', 'Gender', 'MaritalStatus']

model_clean, X_test_clean, y_pred_clean = train_clean_model(
    X_train, X_test, y_train, y_test,
    features_to_remove=FEATURES_TO_REMOVE,
    pos_weight=pos_weight,
    threshold=MY_THRESHOLD,
)
plot_confusion_matrix(y_test, y_pred_clean,
                      title='Clean model: protected features removed')

In [ ]:
log_run(
    project=WANDB_PROJECT,
    run_name='04_clean_features',
    model=model_clean,
    X_test=X_test_clean,
    y_test=y_test,
    y_pred=y_pred_clean,
    config={
        'scale_pos_weight': round(pos_weight, 1),
        'threshold': MY_THRESHOLD,
        'features_removed': FEATURES_TO_REMOVE,
    },
)

---

## Section 7: Review Your Runs in WandB

*About 25 minutes.*

Open your WandB project in the browser. You should have four runs:

| Run | What changed |
|:--|:--|
| 01_naive | Default XGBoost, threshold 0.5 |
| 02_balanced | Added scale_pos_weight |
| 03_threshold | Adjusted threshold to your chosen value |
| 04_clean_features | Removed protected and high-risk features |

For each run, write one sentence: what changed and what was the effect on false negatives. You are building the narrative you would give to the HR director.

Be ready to answer: which single change made the biggest difference, and what did it cost?

In [ ]:
# Print your WandB project URL
print(f'https://wandb.ai/{wandb.api.default_entity}/{WANDB_PROJECT}')

<details>
<summary>Who actually owns the deployment decision in a real organisation?</summary>

Not the ML engineer alone, and not the HR director alone.

In practice, a model that influences employment decisions will involve the ML engineer who built and evaluated it, the HR director or people team who commissioned it, a legal or compliance function who reviewed it against the Equality Act and UK GDPR, and often a senior sign-off from whoever holds accountability for employment practices.

The ML engineer's specific accountability is the technical choices: the threshold, the features included, the evaluation methodology, the known failure modes. If you deploy a model and later it emerges that you knew the false negative rate was high or that protected characteristics were indirectly encoded, 'the HR director approved it' is not a sufficient defence.

This is what K26 and B4 mean in practice: knowing the ethical dimensions of your technical decisions and being able to account for them.

</details>

---

## Section 8: Wrap Up

*About 30 minutes.*

The HR director is asking for a recommendation. You have four runs, a justified threshold, and a feature list you can defend. Answer the questions below before the session closes.

You do not need to write in the notebook. These are the conversation questions.

**1. Which model would you recommend deploying, and why?**

Name the run. State the threshold. Say which metric drove your decision and why that metric maps to the actual cost of failure in this domain.

---

**2. Who bears the false negative in this system?**

An employee who is at risk of leaving but the model does not flag them. What happens to them? What happens to the organisation?

---

**3. Why did you remove Age, Gender, and MaritalStatus?**

Not because someone told you to. Because you can explain the legal basis, the performance impact, and the professional judgment behind the decision.

---

**4. What would make you withdraw this model after deployment?**

Name a specific condition: a metric threshold, a demographic audit result, a business change. Decide it now, before the pressure to keep it running makes it harder to think clearly.

---

## What you can do now

- Your WandB project is yours permanently. Share it with your manager or mentor as evidence of your model evaluation practice.
- Add a note to your learning journal: what was the most uncomfortable decision you made today and why?
- The IBM attrition dataset is widely used. If you want to go further, try adding cross-validation or a SMOTE *(see glossary.md)* resampling approach and log those runs to the same project.

*This workshop maps to K26 (ethical aspects of data and ML), B4 (acts with integrity), S11 (data analysis and model development), and K7 (performance evaluation).*